# Compound extremes in FOSI

In [ ]:
# general use packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# packages for altering time to match up!
import sys
import cftime

# climpred packages
# import climpred
# from climpred import HindcastEnsemble
# from climpred.tutorial import load_dataset
# from climpred.stats import rm_poly

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat

import cartopy.crs as ccrs
import cartopy.feature as cfeature

## load data

In [ ]:
depth = 'surface' # surface, 100m_mean

In [ ]:
sst_ext = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/TEMP.monthly.0.9.regrid.extremes.' + depth + '.nc')['TEMP']
ph_ext = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/pH_3D.monthly.0.9.regrid.extremes.' + depth + '.nc')['pH_3D']
o2_ext = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/O2.monthly.0.1.regrid.extremes.' + depth + '.nc')['O2']

In [ ]:
# biol. extremes
thold = '0.9'
# bio_high = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/photoC_tot_zint_100m.monthly.' + thold + '.regrid.extremes.nc')['photoC_tot_zint_100m']
biomass_high = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.monthly.' + thold + '.regrid.extremes.update.log.nc')['totC_zint_100m']
# chl_high = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totChl_zint_100m.monthly.' + thold + '.regrid.extremes.nc')['totChl_zint_100m']

thold = '0.1'
# bio_low = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/photoC_tot_zint_100m.monthly.' + thold + '.regrid.extremes.nc')['photoC_tot_zint_100m']
biomass_low = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.monthly.' + thold + '.regrid.extremes.update.log.nc')['totC_zint_100m']
# chl_low = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totChl_zint_100m.monthly.' + thold + '.regrid.extremes.nc')['totChl_zint_100m']

## Save data and Figure S6

In [ ]:
biomass_high_sst = (biomass_high.where(sst_ext == 1) > 0)
biomass_high_ph  = (biomass_high.where(ph_ext == 1) > 0)
biomass_high_o2  = (biomass_high.where(o2_ext  == 1) > 0)

In [ ]:
biomass_low_sst = (biomass_low.where(sst_ext == 1) > 0)
biomass_low_ph  = (biomass_low.where(ph_ext == 1) > 0)
biomass_low_o2  = (biomass_low.where(o2_ext  == 1) > 0)

In [ ]:
# mapping
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sea_ice = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/LR/IFRAC.regrid.nc')['IFRAC']
sea_ice = sea_ice.mean('time')
ice_mask = (np.isnan(sea_ice.where(sea_ice > 0)))
# calculate the percentage that do overlap!

f, ax = plt.subplots(2,3,figsize=(14,5),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))
ax = ax.flatten()

## high
(biomass_high_sst.sum('time') / biomass_high.sum('time') * 100).where(ice_mask).plot(ax = ax[0 + 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')

(biomass_high_ph.sum('time') / biomass_high.sum('time') * 100).where(ice_mask).plot(ax = ax[1 + 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')

(biomass_high_o2.sum('time') / biomass_high.sum('time') * 100).where(ice_mask).plot(ax = ax[2 + 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')


## low
(biomass_low_sst.sum('time') / biomass_low.sum('time') * 100).where(ice_mask).plot(ax = ax[3 - 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')

(biomass_low_ph.sum('time') / biomass_low.sum('time') * 100).where(ice_mask).plot(ax = ax[4 - 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')

im = (biomass_low_o2.sum('time') / biomass_low.sum('time') * 100).where(ice_mask).plot(ax = ax[5 - 3],transform = ccrs.PlateCarree(),add_colorbar = False,levels=np.arange(0,110,10),cmap='Reds')

for i in range(len(ax)):
    ax[i].add_feature(cfeature.LAND,color='k',zorder=10)
    ax[i].set_global()

plt.subplots_adjust(wspace=0.05, hspace=0.01)

f.savefig('./figures/SF6.pdf')

In [ ]:
f, ax = plt.subplots(1,1,figsize=(13,9),subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

# f.subplots_adjust(right=0.8)
# cbar_ax = f.add_axes([0.85, 0.25, 0.025, 0.50])
cbar_ax = f.add_axes([0.15, 0.25, 0.8, 0.05])

cbar = f.colorbar(im, cax=cbar_ax, ticks=[0,25,50,75,100], orientation = 'horizontal')
cbar.ax.tick_params(labelsize=10)
cbar.set_label('', rotation=270,fontsize=14)

f.savefig('./figures/clean/percentage.colorbar.horizontal.pdf')

In [ ]:
biomass_high_sst.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.TEMP.monthly.0.9.regrid.extremes.final.nc')
biomass_high_ph.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.pH_3D.monthly.0.9.regrid.extremes.final.nc')
biomass_high_o2.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.O2.monthly.0.9.regrid.extremes.final.nc')

In [ ]:
biomass_low_sst.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.TEMP.monthly.0.1.regrid.extremes.final.nc')
biomass_low_ph.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.pH_3D.monthly.0.1.regrid.extremes.final.nc')
biomass_low_o2.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/totC_zint_100m.O2.monthly.0.1.regrid.extremes.final.nc')